# Notebook 06 — QA generativo controllato su dataset Amazon-like

Questo notebook ricrea il sistema QA usando il dataset Amazon-like salvato nelle cartelle:

```text
data/processed/test_dataset2_intersection
data/processed/test_dataset2_recommender
```

L'obiettivo non è addestrare un LLM e non usa API esterne.  
Il sistema genera una risposta nuova ma controllata, usando evidenze recuperate da:

- metadati prodotto;
- coppie domanda/risposta esistenti;
- recensioni testuali.

La logica segue l'idea del product-aware QA: la risposta deve essere fondata su recensioni e attributi del prodotto, non inventata liberamente.

## 1. Import librerie

Carichiamo librerie standard, pandas, pyarrow e scikit-learn.  
Il notebook usa TF-IDF per recuperare evidenze rilevanti e template controllati per generare la risposta finale.

In [1]:
#Importiamo le librerie usate nel notebook
#Non vengono usate API esterne o modelli online
#Il sistema lavora solo sui file locali del progetto

from pathlib import Path
from IPython.display import display
import json
import html
import math
import re
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

## 2. Percorsi del progetto

La root è fissata esplicitamente, come negli altri notebook Amazon-like.  
Il notebook non deve creare una cartella `data` dentro `notebooks/test_dataset2`.

In [2]:
#Impostiamo manualmente la root del progetto
#Usiamo le stesse cartelle del dataset Amazon-like usate dagli altri notebook
#In questo modo evitiamo di creare una cartella data dentro test_dataset2

PROJECT_ROOT=Path(r"C:\Users\arsid\Desktop\Tesi-ecommerce")

DATA_DIR=PROJECT_ROOT/"data"
PROCESSED_DIR=DATA_DIR/"processed"

INTERSECTION_DIR=PROCESSED_DIR/"test_dataset2_intersection"
RECOMMENDER_DIR=PROCESSED_DIR/"test_dataset2_recommender"
OUTPUT_DIR=RECOMMENDER_DIR/"notebook06_qa"

ITEMS_PATH=INTERSECTION_DIR/"items.parquet"
QA_PATH=INTERSECTION_DIR/"qa.parquet"
RATINGS_PATH=INTERSECTION_DIR/"ratings.parquet"

OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

print("Project root:",PROJECT_ROOT)
print("Intersection dir:",INTERSECTION_DIR)
print("Recommender dir:",RECOMMENDER_DIR)
print("Output dir:",OUTPUT_DIR)
print("Items path:",ITEMS_PATH)
print("QA path:",QA_PATH)
print("Ratings path:",RATINGS_PATH)

Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Intersection dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_intersection
Recommender dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_recommender
Output dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_recommender\notebook06_qa
Items path: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_intersection\items.parquet
QA path: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_intersection\qa.parquet
Ratings path: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_intersection\ratings.parquet


## 3. Parametri principali

Questi parametri controllano quante review usare per ogni prodotto e quanto deve essere forte l'evidenza per generare una risposta.

In [3]:
#Definiamo soglie e limiti del sistema QA
#Valori troppo bassi aumentano le risposte ma anche il rischio di risposte deboli
#Valori troppo alti rendono il sistema troppo prudente

RANDOM_STATE=42

MAX_REVIEWS_PER_PRODUCT=3000
MAX_REVIEW_SENTENCES=3500
MAX_QA_PER_PRODUCT=200
MAX_METADATA_ROWS=20

MIN_SENTENCE_WORDS=5
MAX_SENTENCE_WORDS=45

MIN_QA_SCORE=0.16
MIN_REVIEW_SCORE=0.08
MIN_METADATA_SCORE=0.06
MIN_EVIDENCE_COUNT=2

TOP_EVIDENCE=8
MAX_EVIDENCES_USED=6

TFIDF_MAX_FEATURES=30000

## 4. Caricamento dei dati

Carichiamo prodotti, QA e recensioni.  
Per le recensioni leggiamo solo le colonne necessarie: `parent_asin`, `rating`, `review_title`, `review_text`.

In [4]:
#Carichiamo i file prodotti dai notebook precedenti
#Per ratings.parquet leggiamo solo le colonne necessarie al QA
#Se review_text non esiste il notebook si ferma perché non può usare le recensioni

required_paths=[ITEMS_PATH,QA_PATH,RATINGS_PATH]
missing_paths=[path for path in required_paths if not path.exists()]

if missing_paths:
    raise FileNotFoundError("File mancanti: "+str(missing_paths))

items_df=pd.read_parquet(ITEMS_PATH)
qa_df=pd.read_parquet(QA_PATH)

ratings_schema=pq.ParquetFile(RATINGS_PATH).schema_arrow.names
wanted_rating_columns=["parent_asin","rating","review_title","review_text"]
available_rating_columns=[col for col in wanted_rating_columns if col in ratings_schema]

if "review_text" not in available_rating_columns:
    raise ValueError("review_text non è presente in ratings.parquet. Riesegui il Notebook 01 con SAVE_TEXT_OFREVIEW=True")

ratings_df=pd.read_parquet(RATINGS_PATH,columns=available_rating_columns)

print("items_df:",items_df.shape)
print("qa_df:",qa_df.shape)
print("ratings_df:",ratings_df.shape)
print("Colonne ratings:",list(ratings_df.columns))

items_df: (32266, 16)
qa_df: (256988, 12)
ratings_df: (5245310, 4)
Colonne ratings: ['parent_asin', 'rating', 'review_title', 'review_text']


## 5. Controlli di copertura

Verifichiamo quanti prodotti hanno metadati, QA e recensioni.  
Nel tuo dataset Amazon-like dovrebbero essere tutti coperti.

In [5]:
#Controlliamo la copertura delle tre fonti informative
#Il sistema QA funziona meglio quando ogni prodotto ha QA e review
#Questi numeri aiutano a capire se i file sono coerenti

stats_rows=[]

stats_rows.append({"dataset":"items","rows":len(items_df),"products":items_df["parent_asin"].nunique()})
stats_rows.append({"dataset":"qa","rows":len(qa_df),"products":qa_df["parent_asin"].nunique()})
stats_rows.append({"dataset":"ratings_reviews","rows":len(ratings_df),"products":ratings_df["parent_asin"].nunique()})

review_text_series=ratings_df["review_text"].dropna().astype(str).str.strip()
review_text_series=review_text_series[review_text_series!=""]

stats_rows.append({
    "dataset":"non_empty_review_text",
    "rows":len(review_text_series),
    "products":ratings_df.loc[ratings_df["review_text"].notna(),"parent_asin"].nunique()
})

coverage_df=pd.DataFrame(stats_rows)
display(coverage_df)

,dataset,rows,products
0,items,32266,32266
1,qa,256988,32266
2,ratings_reviews,5245310,32266
3,non_empty_review_text,5244869,32266


## 6. Funzioni di pulizia testuale

Le recensioni Amazon contengono spesso HTML, `<br />`, entità come `&#34;` e testi molto corti.  
Questa sezione pulisce il testo prima di usarlo come evidenza.

In [6]:
#Definiamo funzioni per normalizzare e pulire testo
#Rimuoviamo HTML, caratteri rumorosi e spazi duplicati
#Queste funzioni vengono usate per QA, recensioni e metadati

def to_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value)

def clean_text(value):
    text=to_text(value)
    text=html.unescape(text)
    text=re.sub(r"<br\s*/?>"," ",text,flags=re.IGNORECASE)
    text=re.sub(r"<[^>]+>"," ",text)
    text=re.sub(r"\bbr\b"," ",text,flags=re.IGNORECASE)
    text=re.sub(r"&#34;|&quot;"," ",text,flags=re.IGNORECASE)
    text=re.sub(r"\s+"," ",text)
    text=text.strip()
    return text

def word_count(text):
    return len(re.findall(r"[A-Za-z0-9]+",clean_text(text)))

def is_useless_short_text(text):
    text=clean_text(text).lower()
    useless={
        "good","ok","okay","great","perfect","nice","thanks","thank you",
        "five stars","four stars","three stars","two stars","one star",
        "na","n/a","none"
    }
    if text in useless:
        return True
    if len(text)<=3:
        return True
    return False

def split_sentences(text):
    text=clean_text(text)
    pieces=re.split(r"(?<=[.!?])\s+|\n+|\s+-\s+",text)
    sentences=[]
    seen=set()
    for piece in pieces:
        sentence=clean_text(piece)
        wc=word_count(sentence)
        key=sentence.lower()
        if wc<MIN_SENTENCE_WORDS:
            continue
        if wc>MAX_SENTENCE_WORDS:
            continue
        if is_useless_short_text(sentence):
            continue
        if key in seen:
            continue
        seen.add(key)
        sentences.append(sentence)
    return sentences

## 7. Parsing dei metadati prodotto

I metadati come `features_json`, `description_json`, `categories_json` e `details_json` possono contenere liste o dizionari salvati come stringhe JSON.  
Li trasformiamo in testo leggibile per usarli come evidenza.

In [7]:
#Convertiamo campi JSON dei metadati in testo
#I metadati non sostituiscono le recensioni ma aiutano per domande tecniche
#Limitiamo il numero di elementi per evitare testi troppo lunghi

def flatten_json_value(obj,max_items=12):
    pieces=[]

    def visit(value):
        if len(pieces)>=max_items:
            return
        if value is None:
            return
        if isinstance(value,dict):
            for key,val in value.items():
                if len(pieces)>=max_items:
                    break
                key_text=clean_text(key)
                val_text=clean_text(val) if not isinstance(val,(dict,list,tuple)) else ""
                if key_text and val_text:
                    pieces.append(key_text+": "+val_text)
                else:
                    visit(val)
        elif isinstance(value,(list,tuple)):
            for item in value:
                if len(pieces)>=max_items:
                    break
                visit(item)
        else:
            text=clean_text(value)
            if text:
                pieces.append(text)

    visit(obj)
    return " ".join(pieces)

def json_field_to_text(value,max_items=12):
    text=to_text(value)
    if not text:
        return ""
    try:
        obj=json.loads(text)
        return flatten_json_value(obj,max_items=max_items)
    except Exception:
        return clean_text(text)

def get_item_info(parent_asin):
    key=str(parent_asin)
    item=items_by_asin.get(key,{})
    return item

def item_field(item,col):
    if not item:
        return ""
    return clean_text(item.get(col,""))

## 8. Indici leggeri per prodotto

Creiamo dizionari per recuperare rapidamente QA e recensioni di un singolo prodotto.  
Non costruiamo un indice globale di tutte le recensioni perché sarebbe troppo pesante.

In [8]:
#Costruiamo indici per parent_asin
#groupby.indices non copia tutti i testi ma salva gli indici delle righe
#Questo rende più veloce recuperare QA e review di un prodotto

items_by_asin=items_df.drop_duplicates("parent_asin").set_index("parent_asin").to_dict("index")
qa_by_asin=qa_df.groupby("parent_asin",sort=False).indices
reviews_by_asin=ratings_df.groupby("parent_asin",sort=False).indices

review_evidence_cache={}
qa_evidence_cache={}
metadata_evidence_cache={}

print("Prodotti in items_by_asin:",len(items_by_asin))
print("Prodotti in qa_by_asin:",len(qa_by_asin))
print("Prodotti in reviews_by_asin:",len(reviews_by_asin))

Prodotti in items_by_asin: 32266
Prodotti in qa_by_asin: 32266
Prodotti in reviews_by_asin: 32266


## 9. Intent principali delle domande

Dai tuoi dati le domande più frequenti riguardano compatibilità, funzionamento, fit, accessori inclusi, batteria, qualità, problemi, audio/video e facilità d'uso.  
Usiamo regole semplici per riconoscere questi intenti.

In [9]:
#Definiamo parole chiave per riconoscere il tipo di domanda
#Gli intenti derivano dalle frequenze viste nel dataset Amazon-like
#Questa parte serve a recuperare evidenze più pertinenti dalle recensioni

INTENT_TERMS={
    "compatibility":{
        "work","works","compatible","support","supports","use","using",
        "windows","mac","iphone","ipad","android","samsung","galaxy",
        "canon","nikon","dell","hp","laptop","computer","phone","tv",
        "router","camera","printer","xbox","playstation","ps3","ps4"
    },
    "fit":{
        "fit","fits","size","case","screen","protector","cover","mount",
        "holder","adapter","inch","inches"
    },
    "included":{
        "come","comes","include","includes","included","with","cable",
        "adapter","charger","battery","batteries","remote","manual",
        "accessory","accessories"
    },
    "quality":{
        "good","great","quality","worth","recommend","recommended",
        "buy","purchase","better","best","price","value","durable"
    },
    "ease":{
        "easy","install","installation","setup","set","use","using",
        "simple","instructions"
    },
    "battery":{
        "battery","charge","charging","last","lasts","hours","life",
        "power","drain"
    },
    "problems":{
        "problem","problems","issue","issues","broke","broken","stopped",
        "defective","return","returned","fail","failed","doesn","didn",
        "waste","bad"
    },
    "audio_video":{
        "sound","audio","speaker","headphone","picture","video","screen",
        "display","resolution","quality","hdmi","blu","ray"
    },
    "general":{
        "product","item","works","good","use","quality"
    }
}

DIRECT_METADATA_PATTERNS={
    "rating":{"rating","ratings","stars","average rating","avg rating"},
    "price":{"price","cost","expensive","cheap"},
    "category":{"category","kind","type"},
    "title":{"name","title","called"},
    "store":{"brand","store","seller","manufacturer"}
}

STOP_WORDS={
    "the","a","an","is","it","this","that","to","for","of","and","or",
    "with","on","in","does","do","will","can","would","i","you","my",
    "your","be","are","was","were","as","at","by","from","about"
}

POSITIVE_WORDS={
    "good","great","excellent","perfect","works","worked","fine",
    "easy","recommend","recommended","love","loved","best","better",
    "quality","durable","useful","solid","amazing","satisfied",
    "worth","reliable","nice","well","fast","happy"
}

NEGATIVE_WORDS={
    "bad","poor","problem","problems","issue","issues","broke",
    "broken","stopped","defective","return","returned","waste",
    "disappointed","worse","worst","hard","difficult","failed",
    "fail","noisy","cheap","weak","dead"
}

NEGATIVE_PHRASES=[
    "does not work","doesn't work","did not work","didn't work",
    "stopped working","not compatible","not fit","does not fit",
    "doesn't fit","would not recommend","do not recommend"
]

POSITIVE_PHRASES=[
    "works great","works fine","works perfectly","worked great",
    "easy to use","easy to install","highly recommend","good quality",
    "great product","great price","battery life"
]

def tokenize_text(text):
    text=clean_text(text).lower()
    return set(re.findall(r"[a-z0-9]+",text))

def detect_question_intent(question):
    question_text=clean_text(question).lower()
    terms=tokenize_text(question_text)
    scores={}

    for intent,intent_terms in INTENT_TERMS.items():
        direct_overlap=len(terms & intent_terms)
        phrase_bonus=0
        for term in intent_terms:
            if " " in term and term in question_text:
                phrase_bonus+=2
        scores[intent]=direct_overlap+phrase_bonus

    best_intent=max(scores,key=scores.get)
    if scores[best_intent]==0:
        return "general"
    return best_intent

def build_expanded_query(question,intent):
    intent_terms=sorted(INTENT_TERMS.get(intent,set()))
    return clean_text(question)+" "+" ".join(intent_terms[:25])

## 10. Risposte dirette dai metadati

Alcune domande non devono usare recensioni: rating medio, prezzo, categoria e nome del prodotto sono già nei metadati.

In [10]:
#Gestiamo domande semplici usando direttamente i metadati
#Questo evita di cercare nelle recensioni informazioni già strutturate
#Se la domanda non è metadata-based restituiamo None

def detect_metadata_request(question):
    text=clean_text(question).lower()
    for request_type,patterns in DIRECT_METADATA_PATTERNS.items():
        for pattern in patterns:
            if pattern in text:
                return request_type
    return None

def answer_from_metadata(parent_asin,question):
    request_type=detect_metadata_request(question)
    if request_type is None:
        return None

    item=get_item_info(parent_asin)
    if not item:
        return None

    title=item_field(item,"title")
    avg_rating=item.get("avg_rating",np.nan)
    num_ratings=item.get("num_ratings",np.nan)
    price=item.get("price",np.nan)
    category=item_field(item,"category")
    source_category=item_field(item,"source_category")
    store=item_field(item,"store")

    if request_type=="rating":
        if pd.notna(avg_rating):
            answer=f"The product has an average rating of {avg_rating} based on {num_ratings} ratings."
        else:
            answer="I did not find a reliable average rating in the product metadata."
    elif request_type=="price":
        if pd.notna(price) and str(price)!="nan":
            answer=f"The product price stored in the metadata is {price}."
        else:
            answer="I did not find a reliable price in the product metadata."
    elif request_type=="category":
        category_text=category or source_category
        if category_text:
            answer=f"The product is listed under: {category_text}."
        else:
            answer="I did not find a reliable category in the product metadata."
    elif request_type=="store":
        if store:
            answer=f"The product store or brand field is: {store}."
        else:
            answer="I did not find a reliable store or brand field in the product metadata."
    elif request_type=="title":
        if title:
            answer=f"The product title is: {title}."
        else:
            answer="I did not find a reliable product title in the metadata."
    else:
        return None

    return {
        "parent_asin":str(parent_asin),
        "title":title,
        "question":question,
        "intent":"metadata_"+request_type,
        "answer":answer,
        "confidence":"high",
        "source":"product_metadata",
        "top_score":1.0,
        "evidence_count":1,
        "positive_evidence":0,
        "negative_evidence":0,
        "mixed_evidence":0,
        "warning":"metadata answer"
    }

## 11. Costruzione delle evidenze da QA

Le QA esistenti rimangono importanti quando la domanda dell'utente è simile a una domanda già fatta.  
Però non sono più l'unica fonte: servono come evidenza forte insieme alle recensioni.

In [11]:
#Costruiamo evidenze QA per un prodotto
#Ogni riga contiene domanda e risposta storica
#Usiamo una cache per non ricostruire le stesse evidenze più volte

def get_product_qa_rows(parent_asin):
    key=str(parent_asin)
    idx=qa_by_asin.get(key)
    if idx is None:
        return pd.DataFrame(columns=qa_df.columns)
    product_qa_df=qa_df.iloc[idx].copy()
    if len(product_qa_df)>MAX_QA_PER_PRODUCT:
        product_qa_df=product_qa_df.sample(MAX_QA_PER_PRODUCT,random_state=RANDOM_STATE)
    return product_qa_df

def build_qa_evidence(parent_asin):
    key=str(parent_asin)
    if key in qa_evidence_cache:
        return qa_evidence_cache[key]

    product_qa_df=get_product_qa_rows(key)
    rows=[]

    for row in product_qa_df.itertuples(index=False):
        question=clean_text(getattr(row,"question",""))
        answer=clean_text(getattr(row,"answer",""))
        if not question and not answer:
            continue
        evidence_text=(question+" "+answer).strip()
        if word_count(evidence_text)<3:
            continue
        rows.append({
            "parent_asin":key,
            "source_type":"qa_pair",
            "question_text":question,
            "answer_text":answer,
            "evidence_text":evidence_text,
            "rating":np.nan
        })

    evidence_df=pd.DataFrame(rows)
    qa_evidence_cache[key]=evidence_df
    return evidence_df

## 12. Costruzione delle evidenze dalle recensioni

Questa è la parte che aumenta davvero la copertura.  
Per ogni prodotto estraiamo frasi dalle review e poi cerchiamo quelle più pertinenti alla domanda.

In [12]:
#Costruiamo evidenze testuali dalle recensioni del prodotto
#Campioniamo se un prodotto ha troppe recensioni per mantenere il notebook gestibile
#Ogni frase viene trattata come possibile evidenza per rispondere

def get_product_reviews(parent_asin):
    key=str(parent_asin)
    idx=reviews_by_asin.get(key)
    if idx is None:
        return pd.DataFrame(columns=ratings_df.columns)

    product_reviews_df=ratings_df.iloc[idx].copy()
    product_reviews_df=product_reviews_df.dropna(subset=["review_text"])
    product_reviews_df=product_reviews_df[product_reviews_df["review_text"].astype(str).str.strip()!=""]

    if len(product_reviews_df)>MAX_REVIEWS_PER_PRODUCT:
        product_reviews_df=product_reviews_df.sample(MAX_REVIEWS_PER_PRODUCT,random_state=RANDOM_STATE)

    return product_reviews_df

def build_review_evidence(parent_asin):
    key=str(parent_asin)
    if key in review_evidence_cache:
        return review_evidence_cache[key]

    product_reviews_df=get_product_reviews(key)
    rows=[]
    seen=set()

    for row in product_reviews_df.itertuples(index=False):
        rating=getattr(row,"rating",np.nan)
        review_title=clean_text(getattr(row,"review_title",""))
        review_text=clean_text(getattr(row,"review_text",""))
        combined_text=(review_title+". "+review_text).strip()
        sentences=split_sentences(combined_text)

        for sentence in sentences:
            sentence_key=sentence.lower()
            if sentence_key in seen:
                continue
            seen.add(sentence_key)
            rows.append({
                "parent_asin":key,
                "source_type":"review_sentence",
                "question_text":"",
                "answer_text":"",
                "evidence_text":sentence,
                "rating":rating
            })
            if len(rows)>=MAX_REVIEW_SENTENCES:
                break

        if len(rows)>=MAX_REVIEW_SENTENCES:
            break

    evidence_df=pd.DataFrame(rows)
    review_evidence_cache[key]=evidence_df
    return evidence_df

## 13. Evidenze dai metadati descrittivi

Aggiungiamo titolo, feature, descrizione, categorie e dettagli come evidenze brevi.  
Sono utili per domande su compatibilità, contenuto della confezione o caratteristiche tecniche.

In [13]:
#Costruiamo evidenze dai metadati descrittivi
#Queste evidenze sono poche ma spesso più tecniche delle recensioni
#Usiamo una cache anche per i metadati

def build_metadata_evidence(parent_asin):
    key=str(parent_asin)
    if key in metadata_evidence_cache:
        return metadata_evidence_cache[key]

    item=get_item_info(key)
    rows=[]

    if not item:
        evidence_df=pd.DataFrame(rows)
        metadata_evidence_cache[key]=evidence_df
        return evidence_df

    metadata_fields=[
        ("title",item_field(item,"title")),
        ("category",item_field(item,"category")),
        ("features",json_field_to_text(item.get("features_json",""),max_items=12)),
        ("description",json_field_to_text(item.get("description_json",""),max_items=12)),
        ("categories",json_field_to_text(item.get("categories_json",""),max_items=12)),
        ("details",json_field_to_text(item.get("details_json",""),max_items=12))
    ]

    for field_name,text in metadata_fields:
        text=clean_text(text)
        if not text:
            continue
        if word_count(text)<2:
            continue
        rows.append({
            "parent_asin":key,
            "source_type":"metadata_"+field_name,
            "question_text":"",
            "answer_text":"",
            "evidence_text":text,
            "rating":np.nan
        })

    evidence_df=pd.DataFrame(rows)
    metadata_evidence_cache[key]=evidence_df
    return evidence_df

## 14. Retrieval multi-sorgente

Per una domanda, recuperiamo evidenze da QA, recensioni e metadati.  
La similarità TF-IDF viene combinata con un piccolo punteggio di keyword overlap basato sull'intent.

In [14]:
#Recuperiamo evidenze usando TF-IDF e overlap lessicale
#La query viene espansa con parole dell'intent riconosciuto
#Il risultato è una tabella ordinata per rilevanza

def keyword_overlap_score(question,intent,text):
    question_terms=tokenize_text(question)-STOP_WORDS
    text_terms=tokenize_text(text)-STOP_WORDS
    intent_terms=INTENT_TERMS.get(intent,set())

    if not text_terms:
        return 0.0

    question_overlap=len(question_terms & text_terms)/max(1,len(question_terms))
    intent_overlap=len(intent_terms & text_terms)/max(1,min(len(intent_terms),20))

    score=0.70*question_overlap+0.30*intent_overlap
    return float(min(1.0,score))

def retrieve_evidence(parent_asin,question,top_k=TOP_EVIDENCE):
    intent=detect_question_intent(question)

    qa_evidence_df=build_qa_evidence(parent_asin)
    review_evidence_df=build_review_evidence(parent_asin)
    metadata_evidence_df=build_metadata_evidence(parent_asin)

    evidence_frames=[
        qa_evidence_df,
        review_evidence_df,
        metadata_evidence_df
    ]

    evidence_frames=[frame for frame in evidence_frames if frame is not None and not frame.empty]

    if not evidence_frames:
        return pd.DataFrame(),intent

    evidence_df=pd.concat(evidence_frames,ignore_index=True)
    evidence_df["evidence_text"]=evidence_df["evidence_text"].fillna("").astype(str).map(clean_text)
    evidence_df=evidence_df[evidence_df["evidence_text"].str.len()>0].copy()

    if evidence_df.empty:
        return pd.DataFrame(),intent

    expanded_question=build_expanded_query(question,intent)
    corpus=[expanded_question]+evidence_df["evidence_text"].tolist()

    vectorizer=TfidfVectorizer(
        stop_words="english",
        ngram_range=(1,2),
        max_features=TFIDF_MAX_FEATURES
    )

    matrix=vectorizer.fit_transform(corpus)
    tfidf_scores=cosine_similarity(matrix[0],matrix[1:]).ravel()

    evidence_df["tfidf_score"]=tfidf_scores
    evidence_df["keyword_score"]=evidence_df["evidence_text"].map(lambda text:keyword_overlap_score(question,intent,text))

    source_boost=evidence_df["source_type"].map({
        "qa_pair":0.04,
        "review_sentence":0.00,
        "metadata_title":0.02,
        "metadata_category":0.02,
        "metadata_features":0.03,
        "metadata_description":0.03,
        "metadata_categories":0.02,
        "metadata_details":0.03
    }).fillna(0.0)

    evidence_df["score"]=0.78*evidence_df["tfidf_score"]+0.18*evidence_df["keyword_score"]+source_boost
    evidence_df["score"]=evidence_df["score"].clip(0,1)

    evidence_df=evidence_df.sort_values(["score","tfidf_score","keyword_score"],ascending=False)
    evidence_df=evidence_df.drop_duplicates(subset=["source_type","evidence_text"])
    evidence_df=evidence_df.head(top_k).reset_index(drop=True)

    return evidence_df,intent

## 15. Polarità delle evidenze

Per generare risposte più utili distinguiamo evidenze positive, negative e miste.  
La classificazione è semplice: usa parole/phrase positive o negative e, quando disponibile, il rating della review.

In [15]:
#Classifichiamo ogni evidenza come positiva, negativa o neutra/mista
#Questa euristica non è sentiment analysis avanzata
#Serve solo a controllare la risposta generata e ridurre risposte troppo sicure

def contains_phrase(text,phrases):
    text=clean_text(text).lower()
    return any(phrase in text for phrase in phrases)

def classify_evidence_polarity(text,rating=np.nan):
    text_clean=clean_text(text).lower()
    tokens=tokenize_text(text_clean)

    positive_score=len(tokens & POSITIVE_WORDS)
    negative_score=len(tokens & NEGATIVE_WORDS)

    if contains_phrase(text_clean,POSITIVE_PHRASES):
        positive_score+=2
    if contains_phrase(text_clean,NEGATIVE_PHRASES):
        negative_score+=3

    try:
        rating_value=float(rating)
    except Exception:
        rating_value=np.nan

    if pd.notna(rating_value):
        if rating_value>=4:
            positive_score+=1
        elif rating_value<=2:
            negative_score+=1

    if negative_score>positive_score:
        return "negative"
    if positive_score>negative_score:
        return "positive"
    return "mixed"

def add_polarity(evidence_df):
    if evidence_df.empty:
        evidence_df["polarity"]=[]
        return evidence_df
    evidence_df=evidence_df.copy()
    evidence_df["polarity"]=[
        classify_evidence_polarity(text,rating)
        for text,rating in zip(evidence_df["evidence_text"],evidence_df["rating"])
    ]
    return evidence_df

## 16. Generazione controllata della risposta

Questa sezione genera una risposta nuova, ma vincolata alle evidenze recuperate.  
Se le evidenze sono poche o deboli, il sistema non inventa e usa un fallback.

In [16]:
#Generiamo una risposta controllata a partire dalle evidenze
#La risposta cambia in base all'intent e alla polarità delle evidenze
#Mostriamo sempre una warning per ricordare che la risposta deriva dai dati disponibili

def evidence_threshold(source_type):
    if source_type=="qa_pair":
        return MIN_QA_SCORE
    if source_type=="review_sentence":
        return MIN_REVIEW_SCORE
    if source_type.startswith("metadata_"):
        return MIN_METADATA_SCORE
    return MIN_REVIEW_SCORE

def filter_relevant_evidence(evidence_df):
    if evidence_df.empty:
        return evidence_df
    mask=[
        score>=evidence_threshold(source_type)
        for score,source_type in zip(evidence_df["score"],evidence_df["source_type"])
    ]
    relevant_df=evidence_df.loc[mask].copy()
    relevant_df=relevant_df.head(MAX_EVIDENCES_USED).reset_index(drop=True)
    return relevant_df

def confidence_from_evidence(relevant_df):
    if relevant_df.empty:
        return "none"
    top_score=float(relevant_df["score"].max())
    evidence_count=len(relevant_df)
    qa_count=int((relevant_df["source_type"]=="qa_pair").sum())

    if qa_count>=1 and top_score>=0.24:
        return "high"
    if evidence_count>=4 and top_score>=0.18:
        return "high"
    if evidence_count>=2 and top_score>=0.10:
        return "medium"
    return "low"

def summarize_evidence_counts(relevant_df):
    if relevant_df.empty:
        return 0,0,0
    positive_count=int((relevant_df["polarity"]=="positive").sum())
    negative_count=int((relevant_df["polarity"]=="negative").sum())
    mixed_count=int((relevant_df["polarity"]=="mixed").sum())
    return positive_count,negative_count,mixed_count

def answer_template(intent,positive_count,negative_count,mixed_count,relevant_df):
    evidence_count=len(relevant_df)

    if evidence_count==0:
        return "I did not find enough reliable evidence in the available product data to answer this question."

    if positive_count>negative_count and positive_count>=2:
        dominant="positive"
    elif negative_count>positive_count and negative_count>=2:
        dominant="negative"
    else:
        dominant="mixed"

    if intent in {"compatibility","fit"}:
        if dominant=="positive":
            return "Based on the retrieved QA and reviews, the product seems likely to work or fit for this use case. However, compatibility can depend on the exact device or model, so the answer should be treated as evidence-based rather than definitive."
        if dominant=="negative":
            return "The retrieved evidence suggests possible compatibility or fit problems. I would not give a confident positive answer without checking the exact model and the evidence shown below."
        return "The retrieved evidence is mixed or not specific enough to give a definitive compatibility answer. Some evidence may be useful, but the exact device or model should be checked."

    if intent=="included":
        if dominant=="positive":
            return "The retrieved evidence suggests that the requested accessory or component may be included, but the answer should be checked against the evidence because product bundles can vary."
        if dominant=="negative":
            return "The retrieved evidence suggests that the requested accessory or component may not be included, or that users had doubts about it."
        return "The evidence about what is included is not fully clear. Check the retrieved QA and reviews before relying on the answer."

    if intent=="quality":
        if dominant=="positive":
            return "Based on the retrieved reviews, the product appears to be generally well received. Several pieces of evidence are positive, especially about quality, value, or overall use."
        if dominant=="negative":
            return "The retrieved evidence raises quality concerns. Some users report negative experiences, so the product may not be consistently reliable."
        return "The available evidence is mixed. Some reviews are positive, while others are less clear or negative, so I would avoid a fully confident judgment."

    if intent=="ease":
        if dominant=="positive":
            return "The retrieved reviews suggest that the product is generally easy to use or install."
        if dominant=="negative":
            return "The retrieved reviews suggest possible difficulty with use, installation, or setup."
        return "The evidence about ease of use is mixed or limited."

    if intent=="battery":
        if dominant=="positive":
            return "The retrieved evidence is generally positive about battery or charging behavior, although the exact duration may depend on usage."
        if dominant=="negative":
            return "The retrieved evidence suggests possible battery, charging, or power-related issues."
        return "The available evidence about battery life or charging is not strong enough for a definitive answer."

    if intent=="problems":
        if negative_count>=1:
            return "Yes, some retrieved evidence mentions problems or failures. The issue may not affect every unit, but it is a risk visible in the reviews."
        return "I did not find strong evidence of recurring problems in the retrieved snippets, but this does not prove the product is problem-free."

    if intent=="audio_video":
        if dominant=="positive":
            return "The retrieved evidence is generally positive about audio, video, screen, or picture quality."
        if dominant=="negative":
            return "The retrieved evidence mentions possible audio, video, screen, or picture quality issues."
        return "The evidence about audio or video quality is mixed or limited."

    if dominant=="positive":
        return "Based on the retrieved QA and reviews, the answer is cautiously positive. The response is grounded in the evidence shown below."
    if dominant=="negative":
        return "Based on the retrieved QA and reviews, the answer is cautious or negative. The evidence suggests possible issues."
    return "The retrieved evidence is not strong enough for a clear yes/no answer. The safest interpretation is mixed or uncertain."

def generate_response(parent_asin,question,evidence_df,intent):
    item=get_item_info(parent_asin)
    title=item_field(item,"title")

    evidence_df=add_polarity(evidence_df)
    relevant_df=filter_relevant_evidence(evidence_df)
    relevant_df=add_polarity(relevant_df)

    confidence=confidence_from_evidence(relevant_df)
    positive_count,negative_count,mixed_count=summarize_evidence_counts(relevant_df)

    if relevant_df.empty or (len(relevant_df)<MIN_EVIDENCE_COUNT and confidence!="high"):
        answer="I did not find enough reliable evidence in the available product data to answer this question. Try a more specific question or choose a product with more relevant reviews."
        source="fallback"
        top_score=0.0 if evidence_df.empty else float(evidence_df["score"].max())
    else:
        answer=answer_template(intent,positive_count,negative_count,mixed_count,relevant_df)
        source="qa_reviews_metadata"
        top_score=float(relevant_df["score"].max())

    response={
        "parent_asin":str(parent_asin),
        "title":title,
        "question":question,
        "intent":intent,
        "answer":answer,
        "confidence":confidence,
        "source":source,
        "top_score":round(top_score,4),
        "evidence_count":int(len(relevant_df)),
        "positive_evidence":positive_count,
        "negative_evidence":negative_count,
        "mixed_evidence":mixed_count,
        "warning":"Generated from retrieved evidence, not from an external API"
    }

    return response,relevant_df

## 17. Funzione principale

Questa è la funzione da usare nel notebook o in un futuro backend:

```python
answer_product_question(parent_asin,question)
```

Restituisce una risposta generata e la tabella delle evidenze usate.

In [17]:
#Definiamo la funzione principale del sistema QA
#Prima prova a rispondere dai metadati se la domanda è fattuale
#Altrimenti recupera evidenze da QA, recensioni e metadati e genera una risposta controllata

def answer_product_question(parent_asin,question,top_k=TOP_EVIDENCE):
    parent_asin=str(parent_asin)
    question=clean_text(question)

    if not question:
        raise ValueError("La domanda è vuota")

    if parent_asin not in items_by_asin:
        raise ValueError("parent_asin non trovato in items_df: "+parent_asin)

    metadata_response=answer_from_metadata(parent_asin,question)
    if metadata_response is not None:
        evidence_df=pd.DataFrame([{
            "source_type":"product_metadata",
            "score":1.0,
            "polarity":"mixed",
            "rating":np.nan,
            "evidence_text":metadata_response["answer"]
        }])
        return metadata_response,evidence_df

    evidence_df,intent=retrieve_evidence(parent_asin,question,top_k=top_k)

    if evidence_df.empty:
        response={
            "parent_asin":parent_asin,
            "title":item_field(get_item_info(parent_asin),"title"),
            "question":question,
            "intent":intent,
            "answer":"I did not find evidence for this product.",
            "confidence":"none",
            "source":"fallback",
            "top_score":0.0,
            "evidence_count":0,
            "positive_evidence":0,
            "negative_evidence":0,
            "mixed_evidence":0,
            "warning":"No evidence retrieved"
        }
        return response,evidence_df

    response,relevant_df=generate_response(parent_asin,question,evidence_df,intent)

    if relevant_df.empty:
        evidence_df=add_polarity(evidence_df)
        return response,evidence_df

    return response,relevant_df

## 18. Parent ASIN consigliati per test

La tabella mostra prodotti con molte QA e molte review.  
Sono i migliori per provare domande nuove, perché hanno molte evidenze disponibili.

In [18]:
#Selezioniamo prodotti con molte QA e molte review
#Questi parent_asin sono utili per testare il sistema manualmente
#La colonna copy_code è pronta da copiare nella cella successiva

qa_count=qa_df.groupby("parent_asin").size().rename("qa_count")
review_count=ratings_df.dropna(subset=["review_text"]).groupby("parent_asin").size().rename("review_count")

test_products_df=pd.concat([qa_count,review_count],axis=1).fillna(0).astype(int)
test_products_df["total_evidence"]=test_products_df["qa_count"]+test_products_df["review_count"]
test_products_df=test_products_df.sort_values(["total_evidence","qa_count","review_count"],ascending=False).head(40).reset_index()

titles_df=items_df[["parent_asin","title","source_category","category_group"]].drop_duplicates("parent_asin")
test_products_df=test_products_df.merge(titles_df,on="parent_asin",how="left")
test_products_df["copy_code"]='PARENT_ASIN="'+test_products_df["parent_asin"].astype(str)+'"'

display(test_products_df)

,parent_asin,qa_count,review_count,total_evidence,title,source_category,category_group,copy_code
0,B00BWF5U0M,10,19375,19385,SABRENT 4 Port USB 2.0 Data Hub with Individua...,Electronics,tech,"PARENT_ASIN=""B00BWF5U0M"""
1,B0043T7FXE,10,15943,15953,Logitech M570 Wireless Trackball Mouse – Ergon...,Electronics,tech,"PARENT_ASIN=""B0043T7FXE"""
2,B005X1Y7I2,20,14750,14770,"Anker PowerCore+ Mini, 3350mAh Lipstick-Sized ...",Cell Phones and Accessories,tech,"PARENT_ASIN=""B005X1Y7I2"""
3,B00JRGOKQ8,10,14394,14404,iOttie Easy One Touch 2 Car Mount Holder Unive...,Cell Phones and Accessories,tech,"PARENT_ASIN=""B00JRGOKQ8"""
4,B003NR57BY,10,12632,12642,Logitech M510 Wireless Computer Mouse – Comfor...,Electronics,tech,"PARENT_ASIN=""B003NR57BY"""
5,B00F0DD0I6,10,12557,12567,NETGEAR Nighthawk Smart Wi-Fi Router (R7000) -...,Electronics,tech,"PARENT_ASIN=""B00F0DD0I6"""
6,B00BIFNTMC,10,11319,11329,Anker 2.4G Wireless Vertical Ergonomic Optical...,Electronics,tech,"PARENT_ASIN=""B00BIFNTMC"""
7,B00IVPU786,10,11118,11128,Samsung 32GB up to 48MB/s EVO Class 10 Micro S...,Electronics,tech,"PARENT_ASIN=""B00IVPU786"""
8,B0092KJ9BU,10,11084,11094,Poly Voyager Legend Wireless Headset (Plantron...,Cell Phones and Accessories,tech,"PARENT_ASIN=""B0092KJ9BU"""
9,B00IVPU7AO,10,11010,11020,Samsung EVO 64GB 48/MB/s Micro SDXC Memory Car...,Electronics,tech,"PARENT_ASIN=""B00IVPU7AO"""


## 19. Domande consigliate per test

Queste domande coprono gli intenti più frequenti nei tuoi dati.  
Puoi copiarle nella cella manuale cambiando solo `QUESTION`.

In [19]:
#Creiamo una lista di domande utili per provare diversi intenti
#Sono domande generiche ma frequenti nel dataset Amazon-like
#Non tutte saranno adatte a ogni prodotto

suggested_questions=[
    "Does it work well?",
    "Is it worth buying?",
    "Is it easy to use?",
    "Is it easy to install?",
    "Does it have good quality?",
    "Are there any common problems?",
    "Does it stop working after some time?",
    "Is the battery life good?",
    "Does it come with a cable?",
    "Does it fit well?",
    "Does it work with iPhone?",
    "Does it work with Windows?",
    "Is the sound quality good?",
    "Is the picture quality good?",
    "What is the average rating?"
]

suggested_questions_df=pd.DataFrame({"question":suggested_questions})
display(suggested_questions_df)

,question
0,Does it work well?
1,Is it worth buying?
2,Is it easy to use?
3,Is it easy to install?
4,Does it have good quality?
5,Are there any common problems?
6,Does it stop working after some time?
7,Is the battery life good?
8,Does it come with a cable?
9,Does it fit well?


## 20. Test manuale

Cambia `PARENT_ASIN` e `QUESTION`.  
La risposta viene generata e sotto vengono mostrate le evidenze usate.

In [26]:
#Modifica questi valori per provare il sistema a mano
#PARENT_ASIN identifica il prodotto e deve essere una stringa
#QUESTION è la domanda libera dell'utente

from IPython.display import Markdown

PARENT_ASIN="B003NR57BY"
QUESTION="is it good material?"

response,evidence_df=answer_product_question(PARENT_ASIN,QUESTION,top_k=TOP_EVIDENCE)

answer_text=response.get("answer","")
title_text=response.get("title","")
question_text=response.get("question","")
intent_text=response.get("intent","")
confidence_text=response.get("confidence","")
source_text=response.get("source","")
warning_text=response.get("warning","")
evidence_count=response.get("evidence_count",0)
top_score=response.get("top_score",0.0)

markdown_text=(
    "### Risposta generata\n\n"
    f"**Prodotto:** {title_text}\n\n"
    f"**Domanda:** {question_text}\n\n"
    f"**Risposta:** {answer_text}\n\n"
    "### Dettagli\n\n"
    f"- Intent: `{intent_text}`\n"
    f"- Confidenza: `{confidence_text}`\n"
    f"- Fonte: `{source_text}`\n"
    f"- Evidenze usate: `{evidence_count}`\n"
    f"- Score massimo: `{top_score}`\n"
    f"- Warning: {warning_text}\n"
)

display(Markdown(markdown_text))

summary_df=pd.DataFrame([response]).drop(columns=["answer"],errors="ignore")
display(summary_df)

cols_to_show=[
    "source_type","score","tfidf_score","keyword_score",
    "polarity","rating","evidence_text"
]
available_cols=[col for col in cols_to_show if col in evidence_df.columns]

pd.set_option("display.max_colwidth",1000)
pd.set_option("display.width",200)

if evidence_df.empty:
    display(Markdown("### Evidenze usate\n\nNessuna evidenza recuperata."))
else:
    display(Markdown("### Evidenze usate"))
    evidence_to_show=evidence_df[available_cols].copy()
    evidence_blocks=[]
    for idx,row in evidence_to_show.head(TOP_EVIDENCE).iterrows():
        source=row.get("source_type","")
        score=row.get("score",0.0)
        polarity=row.get("polarity","")
        rating=row.get("rating","")
        text=row.get("evidence_text","")
        evidence_blocks.append(
            f"**Evidenza {len(evidence_blocks)+1}**  \n"
            f"Fonte: `{source}` | Score: `{score:.4f}` | Polarità: `{polarity}` | Rating: `{rating}`  \n"
            f"> {text}"
        )
    display(Markdown("\n\n".join(evidence_blocks)))
    display(evidence_to_show)


### Risposta generata

**Prodotto:** Logitech M510 Wireless Computer Mouse – Comfortable Shape with USB Unifying Receiver, with Back/Forward Buttons and Side-to-Side Scrolling, Dark Gray

**Domanda:** is it good material?

**Risposta:** Based on the retrieved reviews, the product appears to be generally well received. Several pieces of evidence are positive, especially about quality, value, or overall use.

### Dettagli

- Intent: `quality`
- Confidenza: `high`
- Fonte: `qa_reviews_metadata`
- Evidenze usate: `6`
- Score massimo: `0.2024`
- Warning: Generated from retrieved evidence, not from an external API


,parent_asin,title,question,intent,confidence,source,top_score,evidence_count,positive_evidence,negative_evidence,mixed_evidence,warning
0,B003NR57BY,"Logitech M510 Wireless Computer Mouse – Comfortable Shape with USB Unifying Receiver, with Back/Forward Buttons and Side-to-Side Scrolling, Dark Gray",is it good material?,quality,high,qa_reviews_metadata,0.2024,6,6,0,0,"Generated from retrieved evidence, not from an external API"


### Evidenze usate

**Evidenza 1**  
Fonte: `review_sentence` | Score: `0.2024` | Polarità: `positive` | Rating: `5.0`  
> This is a good purchase and a good price.

**Evidenza 2**  
Fonte: `review_sentence` | Score: `0.2001` | Polarità: `positive` | Rating: `5.0`  
> I can not speak to that aspect of it but I can say this is another product and you will agree, a good one at a great price.

**Evidenza 3**  
Fonte: `review_sentence` | Score: `0.1963` | Polarità: `positive` | Rating: `3.0`  
> It's a good mouse, but the material is not as firm as the M450 I purchased 5 years ago.

**Evidenza 4**  
Fonte: `review_sentence` | Score: `0.1695` | Polarità: `positive` | Rating: `5.0`  
> Good buy for size and quality.

**Evidenza 5**  
Fonte: `review_sentence` | Score: `0.1665` | Polarità: `positive` | Rating: `5.0`  
> good price and quality is not bad

**Evidenza 6**  
Fonte: `review_sentence` | Score: `0.1642` | Polarità: `positive` | Rating: `5.0`  
> Totally recommended, very good product.

,source_type,score,tfidf_score,keyword_score,polarity,rating,evidence_text
0,review_sentence,0.202410,0.162754,0.419231,positive,5.0,This is a good purchase and a good price.
1,review_sentence,0.200093,0.159783,0.419231,positive,5.0,"I can not speak to that aspect of it but I can say this is another product and you will agree, a good one at a great price."
2,review_sentence,0.196300,0.084803,0.723077,positive,3.0,"It's a good mouse, but the material is not as firm as the M450 I purchased 5 years ago."
3,review_sentence,0.169500,0.120562,0.419231,positive,5.0,Good buy for size and quality.
4,review_sentence,0.166535,0.116761,0.419231,positive,5.0,good price and quality is not bad
5,review_sentence,0.164230,0.119131,0.396154,positive,5.0,"Totally recommended, very good product."


## 21. Test rapido su più domande

Questa cella prova alcune domande su pochi prodotti consigliati.  
Serve solo come sanity check qualitativo, non come valutazione scientifica finale.

In [21]:
#Eseguiamo un piccolo test automatico su pochi prodotti e domande
#Il test riassume quante domande ricevono una risposta non fallback
#Mostriamo le risposte in tabella senza usare pandas.style

RUN_QUICK_TEST=True

if RUN_QUICK_TEST:
    quick_parent_asins=test_products_df["parent_asin"].head(3).tolist()
    quick_questions=[
        "Does it work well?",
        "Is it worth buying?",
        "Is it easy to use?",
        "Are there any common problems?",
        "Does it come with a cable?"
    ]

    quick_rows=[]

    for parent_asin in quick_parent_asins:
        for question in quick_questions:
            response,_=answer_product_question(parent_asin,question,top_k=TOP_EVIDENCE)
            quick_rows.append({
                "parent_asin":parent_asin,
                "question":question,
                "intent":response["intent"],
                "source":response["source"],
                "confidence":response["confidence"],
                "top_score":response["top_score"],
                "evidence_count":response["evidence_count"],
                "answer":response["answer"]
            })

    quick_results_df=pd.DataFrame(quick_rows)
    pd.set_option("display.max_colwidth",1000)
    pd.set_option("display.width",200)
    display(quick_results_df)

    summary_quick_df=(
        quick_results_df
        .assign(answered=quick_results_df["source"]!="fallback")
        .groupby(["intent","answered"])
        .size()
        .reset_index(name="count")
    )

    display(summary_quick_df)
else:
    print("Quick test disattivato")


,parent_asin,question,intent,source,confidence,top_score,evidence_count,answer
0,B00BWF5U0M,Does it work well?,compatibility,qa_reviews_metadata,high,0.2534,6,"Based on the retrieved QA and reviews, the product seems likely to work or fit for this use case. However, compatibility can depend on the exact device or model, so the answer should be treated as evidence-based rather than definitive."
1,B00BWF5U0M,Is it worth buying?,quality,qa_reviews_metadata,high,0.2876,6,"Based on the retrieved reviews, the product appears to be generally well received. Several pieces of evidence are positive, especially about quality, value, or overall use."
2,B00BWF5U0M,Is it easy to use?,ease,qa_reviews_metadata,high,0.4294,6,The retrieved reviews suggest that the product is generally easy to use or install.
3,B00BWF5U0M,Are there any common problems?,problems,qa_reviews_metadata,medium,0.1598,6,"Yes, some retrieved evidence mentions problems or failures. The issue may not affect every unit, but it is a risk visible in the reviews."
4,B00BWF5U0M,Does it come with a cable?,included,qa_reviews_metadata,high,0.1980,6,"The retrieved evidence suggests that the requested accessory or component may be included, but the answer should be checked against the evidence because product bundles can vary."
5,B0043T7FXE,Does it work well?,compatibility,qa_reviews_metadata,high,0.2557,6,"Based on the retrieved QA and reviews, the product seems likely to work or fit for this use case. However, compatibility can depend on the exact device or model, so the answer should be treated as evidence-based rather than definitive."
6,B0043T7FXE,Is it worth buying?,quality,qa_reviews_metadata,high,0.2537,6,"Based on the retrieved reviews, the product appears to be generally well received. Several pieces of evidence are positive, especially about quality, value, or overall use."
7,B0043T7FXE,Is it easy to use?,ease,qa_reviews_metadata,high,0.5573,6,The retrieved reviews suggest that the product is generally easy to use or install.
8,B0043T7FXE,Are there any common problems?,problems,qa_reviews_metadata,high,0.2350,6,"Yes, some retrieved evidence mentions problems or failures. The issue may not affect every unit, but it is a risk visible in the reviews."
9,B0043T7FXE,Does it come with a cable?,included,qa_reviews_metadata,medium,0.1791,6,"The retrieved evidence suggests that the requested accessory or component may be included, but the answer should be checked against the evidence because product bundles can vary."


,intent,answered,count
0,compatibility,True,3
1,ease,True,3
2,included,True,3
3,problems,True,3
4,quality,True,3


## 22. Salvataggio prodotti testabili

Salviamo la lista dei prodotti consigliati per test futuri.  
Non salviamo un indice globale delle recensioni perché sarebbe pesante e non necessario per il prototipo.

In [22]:
#Salviamo i parent_asin consigliati per test manuali
#Il file è leggero e può essere riutilizzato nel backend o in altri notebook
#Non esportiamo testi completi delle review in questa cella

TEST_PRODUCTS_PATH=OUTPUT_DIR/"qa_test_parent_asins.csv"

test_products_df.to_csv(TEST_PRODUCTS_PATH,index=False)

print("File salvato:",TEST_PRODUCTS_PATH)
print("Righe salvate:",len(test_products_df))

File salvato: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_recommender\notebook06_qa\qa_test_parent_asins.csv
Righe salvate: 40


## 23. Note finali

Questo notebook non addestra un modello neurale generativo.  
La generazione è **controllata e basata su evidenze**: recupera QA, review e metadati, poi compone una risposta con template.

Punti forti:

- usa molte più informazioni rispetto al solo recupero da QA;
- riduce i fallback quando le recensioni contengono evidenze pertinenti;
- mostra le evidenze usate;
- non usa API esterne;
- resta spiegabile per una tesi triennale.

Limiti:

- TF-IDF lavora soprattutto su similarità lessicale;
- la polarità è euristica, non sentiment analysis avanzata;
- la risposta generata non deve essere presentata come verità assoluta;
- per domande molto tecniche o modelli specifici può servire controllo manuale.